# Smart Grid Energy Management — Decision Tree

**Module:** Machine Learning  
**Algorithm:** Decision Tree Regressor (Supervised Learning)  
**Dataset:** Energy Consumption, Generation, Prices & Weather (Spain) — Kaggle  
**Dataset URL:** https://www.kaggle.com/datasets/nicholasjhana/energy-consumption-generation-prices-and-weather

---

## Problem Statement

The objective is to predict the **Actual Electricity Price** (€/MWh) using a **Decision Tree Regressor**. Decision Trees are highly interpretable models that split data into branches to reach a prediction. This notebook follows the same preprocessing pipeline as our other models to ensure a fair comparison.

### Key Design Decisions
| Component | Design |
|---|---|
| **Target Variable** | `price actual` (Continuous) |
| **Predictor Features** | Load, Generation (Solar/Wind/Gas), Temp, Hour, Month, Day of Week |
| **Model** | Decision Tree Regressor |
| **Evaluation** | MAE, RMSE, and R² Score |

## 1. Imports & Style Configuration

In [ ]:
# Standard library
import os
import warnings
warnings.filterwarnings('ignore')

# Numerical / data
import numpy as np
import pandas as pd

# Machine Learning
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Visualisation
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.tree import plot_tree

# Styling
SEED = 42
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})
sns.set_theme(style='whitegrid')

print('Libraries imported successfully.')

## 2. Dataset Loading & Preprocessing

We use the same aggregation and cleaning logic as our RL and Random Forest implementations.

In [ ]:
ENERGY_FILE  = '../data/energy_dataset.csv'
WEATHER_FILE = '../data/weather_features.csv'

energy_df  = pd.read_csv(ENERGY_FILE,  parse_dates=['time'])
weather_df = pd.read_csv(WEATHER_FILE, parse_dates=['dt_iso'])

# Aggregate weather to hourly
weather_agg = (
    weather_df
    .rename(columns={'dt_iso': 'time'})
    .groupby('time')[['temp', 'wind_speed', 'clouds_all']]
    .mean()
    .reset_index()
)

# Handle timezones
energy_df['time'] = pd.to_datetime(energy_df['time'], utc=True).dt.tz_localize(None)
weather_agg['time'] = pd.to_datetime(weather_agg['time'], utc=True).dt.tz_localize(None)

# Merge
df = energy_df.merge(weather_agg, on='time', how='left')

# Feature Selection
core_cols = {
    'price actual': 'price',
    'total load actual': 'load',
    'generation solar': 'solar',
    'generation wind onshore': 'wind',
    'generation fossil gas': 'gas',
    'temp': 'temp'
}

df = df[['time'] + list(core_cols.keys())].rename(columns=core_cols)

# Cleaning
df.dropna(subset=['price', 'load'], inplace=True)
df.fillna(df.median(numeric_only=True), inplace=True)

# Time extraction
df['hour'] = df['time'].dt.hour
df['month'] = df['time'].dt.month
df['day_of_week'] = df['time'].dt.dayofweek

print(f"Dataset Prepared. Shape: {df.shape}")

## 3. Training the Decision Tree

We train a single Decision Tree. Note that we set a `max_depth` to prevent extreme overfitting while maintaining high performance.

In [ ]:
features = ['load', 'solar', 'wind', 'gas', 'temp', 'hour', 'month', 'day_of_week']
X = df[features]
y = df['price']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED)

print("Training Decision Tree Regressor...")
dt_model = DecisionTreeRegressor(max_depth=12, min_samples_split=20, random_state=SEED)
dt_model.fit(X_train, y_train)
print("Training complete.")

## 4. Evaluation & Metrics

In [ ]:
y_pred = dt_model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print(f"MAE: {mae:.4f} EUR/MWh")
print(f"RMSE: {rmse:.4f} EUR/MWh")
print(f"R2 Score: {r2:.4f}")

## 5. Decision Tree Visualization

### Feature Importance
Determining which variables contribute most to the pricing decisions.

In [ ]:
fi_df = pd.DataFrame({'Feature': features, 'Importance': dt_model.feature_importances_}).sort_values(by='Importance', ascending=False)

plt.figure(figsize=(10, 5))
sns.barplot(x='Importance', y='Feature', data=fi_df, palette='magma')
plt.title("Decision Tree Feature Importance")
plt.show()

### Partial Tree Structure
Visualizing the top levels of the tree to understand how it makes high-level decisions.

In [ ]:
plt.figure(figsize=(20, 10))
plot_tree(dt_model, feature_names=features, max_depth=2, filled=True, rounded=True, fontsize=12)
plt.title("Decision Tree Structure (Top 2 Levels)")
plt.show()

### Actual vs Predicted
Comparing the model's predictions against ground truth.

In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(x=y_test, y=y_pred, alpha=0.3, color='purple')
plt.plot([y.min(), y.max()], [y.min(), y.max()], '--r', lw=2)
plt.xlabel("Actual Price (€/MWh)")
plt.ylabel("Predicted Price (€/MWh)")
plt.title("Decision Tree: Actual vs Predicted")
plt.show()